In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
from jax.typing import ArrayLike
import numpyro
import numpyro.distributions as dist

jax.config.update("jax_enable_x64", True)

import altair as alt

from mixres.sim import PoissonEnveloped1D
from mixres.models._inference import run_inference_mcmc

from bspline1d_utils import (
    make_piecewise_bspline_basis,
    make_block_difference_matrix,
    get_segment_diff_sizes,
)
from interval_utils import (
    expand_interval_dataframe,
    compute_interval_widths,
    build_interval_sum_matrix,
)

## Data Generation

In [21]:
# Generate the dataset
left_limits = [0, 0, 5, 8, 10, 15, 18, 20, 25, 30, 35, 40, 50, 60, 70, 80, 90]
right_limits = [5, 12, 15, 25, 20, 30, 35, 40, 45, 50, 55, 60, 70, 80, 90, 95, 100]
intervals = [
    f"[{left},{right}{']' if right in {5, 15, 25, 30, 40, 50, 60, 70, 80, 90, 100} else ')'}"
    for left, right in zip(left_limits, right_limits)
]
print("Intervals:", intervals)

df_obs, df_rates = PoissonEnveloped1D(
    obs_intervals=intervals, n_observations=10, cut_points=[18, 64], seed=42
).generate(return_true_rates=True)
df_expanded = expand_interval_dataframe(df_obs, interval_col="obs_interval", x_col="x")

Intervals: ['[0,5]', '[0,12)', '[5,15]', '[8,25]', '[10,20)', '[15,30]', '[18,35)', '[20,40]', '[25,45)', '[30,50]', '[35,55)', '[40,60]', '[50,70]', '[60,80]', '[70,90]', '[80,95)', '[90,100]']


## Setup

In [22]:
BASIS_DENSITY = 0.8
BOUNDARY_EXT = 0.5
PENALTY_ORDER = 1

In [23]:
df_model = df_obs.groupby(["obs_interval_id"])["y"].agg(N="size", y="sum").reset_index()

x = np.arange(0, 101)
y_obs = df_model["y"].values
interval_id = df_model["obs_interval_id"].values
N = df_model["N"].values

intervals = df_obs["obs_interval"].unique().tolist()
A = build_interval_sum_matrix(intervals)
interval_width = compute_interval_widths(intervals)

# Piecewise spline basis matching DGP cut points [18, 64] -> [0,17], [18,63], [64,100].
# basis_density=0.5 gives 18 points -> 9 basis functions in the first segment.
B, segment_basis_counts = make_piecewise_bspline_basis(
    x,
    cut_points=[18, 64],
    basis_density=BASIS_DENSITY,
    boundary_ext=BOUNDARY_EXT,
    return_basis_counts=True,
)
segment_diff_sizes = get_segment_diff_sizes(segment_basis_counts, order=PENALTY_ORDER)

# Map each x point to its spline segment index (0, 1, 2)
segment_id_x = np.select([x < 18, x < 64], [0, 1], default=2).astype(np.int32)
n_segments = int(segment_id_x.max() + 1)

# Convert to JAX arrays
y_obs = jnp.array(y_obs)
interval_id = jnp.array(interval_id)
N = jnp.array(N)
A_jax = jnp.array(A)
B_jax = jnp.array(B)
interval_width_jax = jnp.array(interval_width)
segment_id_x_jax = jnp.array(segment_id_x)
D_jax = jnp.array(
    make_block_difference_matrix(segment_basis_counts, order=PENALTY_ORDER)
)

In [24]:
def plot_posterior_lam(df_expanded, df_rates, mcmc_samples):
    # Extract posterior samples of lam from MCMC
    lam = mcmc_samples["lam"].mean(axis=0)
    lam_lower = jnp.percentile(mcmc_samples["lam"], 2.5, axis=0)
    lam_upper = jnp.percentile(mcmc_samples["lam"], 97.5, axis=0)

    df_results = df_rates.copy()
    df_results["lam"] = lam
    df_results["lam_lower"] = lam_lower
    df_results["lam_upper"] = lam_upper
    df_results["segment"] = np.select(
        [df_results["x"] < 18, df_results["x"] < 64],
        [0, 1],
        default=2,
    )

    df_obs_lam = (
        df_expanded[["obs_interval", "x", "lambda"]]
        .drop_duplicates(["obs_interval", "x"])
        .groupby("obs_interval", as_index=False)
        .agg(x_min=("x", "min"), x_max=("x", "max"), lambda_obs=("lambda", "mean"))
    )

    # detail="segment:N" draws three discontinuous lines, one per disjoint interval
    true_line = (
        alt.Chart(df_results)
        .mark_line(color="red")
        .encode(x="x", y=alt.Y("lambda:Q", title="lambda"), detail="segment:N")
    )
    obs_lam_lines = (
        alt.Chart(df_obs_lam)
        .mark_rule(color="orange", strokeWidth=2.5, opacity=0.85)
        .encode(
            x=alt.X("x_min:Q"),
            x2="x_max:Q",
            y=alt.Y("lambda_obs:Q", title="lambda"),
        )
    )

    mean_line = (
        alt.Chart(df_results)
        .mark_line(color="blue")
        .encode(x="x", y=alt.Y("lam:Q", title="lambda"), detail="segment:N")
    )
    bands = (
        alt.Chart(df_results)
        .mark_area(color="lightblue", opacity=0.5)
        .encode(
            x="x",
            y=alt.Y("lam_lower:Q", title="lambda"),
            y2=alt.Y2("lam_upper:Q", title="lambda"),
            detail="segment:N",
        )
    )

    return true_line + obs_lam_lines + mean_line + bands

## Experiment 1: B-Spline Regression

In [25]:
def model_bspline(
    A: ArrayLike,
    B: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    segment_id_x: ArrayLike,
    n_segments: int,
    y_obs=None,
):
    """
    Bayesian B-spline regression model with per-segment intercepts.

    Priors:
        beta0  ~ N(0, 1)            global intercept
        alpha  ~ N(0, 1)^S          per-segment intercepts (S = number of segments)
        w      ~ N(0, I_K)          B-spline coefficients

    Likelihood:
        y | beta0, alpha, w ~ Poisson(exp(beta0 + alpha[segment] + B @ w))
    """
    K = B.shape[1]

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    alpha = numpyro.sample("alpha", dist.Normal(0, 1).expand([n_segments]).to_event(1))
    w = numpyro.sample("w", dist.Normal(0, 1).expand([K]).to_event(1))

    # --- Latent rate ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + alpha[segment_id_x] + B @ w))
    lam_interval = (
        jnp.matmul(A, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y_obs)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y_obs)

In [26]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_bspline,
    A=A_jax,
    B=B_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    segment_id_x=segment_id_x_jax,
    n_segments=n_segments,
    y_obs=y_obs,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:00<00:00, 2298.45it/s, 31 steps of size 1.15e-01. acc. prob=0.94]

Number of divergences: 0
dict_keys(['alpha', 'beta0', 'lam', 'w'])


In [27]:
plot_posterior_lam(df_expanded, df_rates, mcmc_samples)

alt.LayerChart(...)

In [38]:
df_w = pd.DataFrame(
    {
        "idx": range(mcmc_samples["w"].shape[1]),
        "w": mcmc_samples["w"].mean(axis=0),
    }
)

points = alt.Chart(df_w).mark_point(size=20, color="red").encode(x="idx", y="w")
lines = alt.Chart(df_w).mark_line(size=1).encode(x="idx", y="w")

points + lines

alt.LayerChart(...)

## Experiment 2: P-Spline Regression

In [11]:
def model_pspline(
    A: ArrayLike,
    B: ArrayLike,
    D: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    segment_diff_sizes: list[int],
    segment_id_x: ArrayLike,
    y_obs: ArrayLike = None,
    epsilon: float = 1e-3,
):
    """
    Bayesian P-spline regression model with segment-wise independent difference priors
    and per-segment intercepts.
    """
    K = B.shape[1]
    n_segments = len(segment_diff_sizes)

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    alpha = numpyro.sample("alpha", dist.Normal(0, 1).expand([n_segments]).to_event(1))

    # Independent difference priors by disjoint spline segment.
    delta_segments = []
    for seg_idx, seg_size in enumerate(segment_diff_sizes):
        delta_seg = numpyro.sample(
            f"delta_seg_{seg_idx}",
            dist.Normal(0, 1).expand([seg_size]).to_event(1),
        )
        delta_segments.append(delta_seg)
    delta = jnp.concatenate(delta_segments)

    w = numpyro.sample("w", dist.Normal(0, 1).expand([K]).to_event(1))

    # Soft constraints on finite differences
    diff = jnp.matmul(D, w)
    penalty = -0.5 * jnp.sum(jnp.square(diff - delta)) / (epsilon**2)
    numpyro.factor("soft_constraint", penalty)

    # --- Latent mean ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + alpha[segment_id_x] + B @ w))
    lam_interval = (
        jnp.matmul(A, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y_obs)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y_obs)

In [39]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_pspline,
    A=A_jax,
    B=B_jax,
    D=D_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    segment_diff_sizes=segment_diff_sizes,
    segment_id_x=segment_id_x_jax,
    y_obs=y_obs,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:09<00:00, 103.74it/s, 1023 steps of size 6.13e-04. acc. prob=0.89]

Number of divergences: 0
dict_keys(['alpha', 'beta0', 'delta_seg_0', 'delta_seg_1', 'delta_seg_2', 'lam', 'w'])


In [40]:
plot_posterior_lam(df_expanded, df_rates, mcmc_samples)

alt.LayerChart(...)

In [41]:
df_w = pd.DataFrame(
    {
        "idx": range(mcmc_samples["w"].shape[1]),
        "w": mcmc_samples["w"].mean(axis=0),
    }
)

points = alt.Chart(df_w).mark_point(size=20, color="red").encode(x="idx", y="w")
lines = alt.Chart(df_w).mark_line(size=1).encode(x="idx", y="w")

points + lines

alt.LayerChart(...)

## Experiment 3: Finnish Horseshoe P-Spline

In [42]:
def model_fHS(
    A: ArrayLike,
    B: ArrayLike,
    D: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    segment_diff_sizes: list[int],
    segment_id_x: ArrayLike,
    y_obs: ArrayLike = None,
    epsilon: float = 1e-3,
    slab_scale: float = 2.0,
    slab_df: float = 4.0,
):
    """
    Bayesian P-spline model with segment-wise independent regularized horseshoe priors
    and per-segment intercepts.

    The regularized (Finnish) horseshoe (Piironen & Vehtari, 2017) adds a Student-t
    slab that clips the tails of the local shrinkage scales, preventing the extreme
    values that cause divergences in HMC with the standard horseshoe.

    For each disjoint spline segment:
        tau_seg  ~ HalfCauchy(1)              global scale
        lambda_seg ~ HalfCauchy(1)            local scales
        c2_seg   ~ InvGamma(slab_df/2,
                             slab_df/2 * slab_scale^2)  slab variance
        lambda_tilde_seg = sqrt(
            c2_seg * lambda_seg^2 / (c2_seg + tau_seg^2 * lambda_seg^2)
        )
        delta_seg = z_seg * tau_seg * lambda_tilde_seg,  z_seg ~ N(0, 1)
    """
    K = B.shape[1]
    n_segments = len(segment_diff_sizes)

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    alpha = numpyro.sample("alpha", dist.Normal(0, 1).expand([n_segments]).to_event(1))

    # Independent regularized horseshoe priors by disjoint spline segment.
    delta_segments = []
    for seg_idx, seg_size in enumerate(segment_diff_sizes):
        # Global and local shrinkage scales
        tau_seg = numpyro.sample(f"tau_seg_{seg_idx}", dist.HalfCauchy(1.0))
        lambda_seg = numpyro.sample(
            f"lambda_seg_{seg_idx}",
            dist.HalfCauchy(1.0).expand([seg_size]).to_event(1),
        )
        # Slab variance: regularizes tails of lambda to prevent divergences
        c2_seg = numpyro.sample(
            f"c2_seg_{seg_idx}",
            dist.InverseGamma(slab_df / 2.0, slab_df / 2.0 * slab_scale**2),
        )
        # Regularized local scale (clipped towards slab_scale when lambda is large)
        lambda_tilde_seg = jnp.sqrt(
            c2_seg * lambda_seg**2 / (c2_seg + tau_seg**2 * lambda_seg**2)
        )
        z_seg = numpyro.sample(
            f"delta_raw_seg_{seg_idx}",
            dist.Normal(0, 1).expand([seg_size]).to_event(1),
        )
        delta_seg = z_seg * tau_seg * lambda_tilde_seg
        delta_segments.append(delta_seg)
    delta = jnp.concatenate(delta_segments)

    w = numpyro.sample("w", dist.Normal(0, 1).expand([K]).to_event(1))

    # Soft constraints on finite differences
    diff = jnp.matmul(D, w)
    penalty = -0.5 * jnp.sum(jnp.square(diff - delta)) / (epsilon**2)
    numpyro.factor("soft_constraint", penalty)

    # --- Latent mean ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + alpha[segment_id_x] + B @ w))
    lam_interval = (
        jnp.matmul(A, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y_obs)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y_obs)

In [43]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_fHS,
    A=A_jax,
    B=B_jax,
    D=D_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    segment_diff_sizes=segment_diff_sizes,
    segment_id_x=segment_id_x_jax,
    y_obs=y_obs,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:16<00:00, 61.24it/s, 1023 steps of size 5.89e-04. acc. prob=0.97]

Number of divergences: 41
dict_keys(['alpha', 'beta0', 'c2_seg_0', 'c2_seg_1', 'c2_seg_2', 'delta_raw_seg_0', 'delta_raw_seg_1', 'delta_raw_seg_2', 'lam', 'lambda_seg_0', 'lambda_seg_1', 'lambda_seg_2', 'tau_seg_0', 'tau_seg_1', 'tau_seg_2', 'w'])


In [44]:
plot_posterior_lam(df_expanded, df_rates, mcmc_samples)

alt.LayerChart(...)

In [45]:
df_w = pd.DataFrame(
    {
        "idx": range(mcmc_samples["w"].shape[1]),
        "w": mcmc_samples["w"].mean(axis=0),
    }
)

points = alt.Chart(df_w).mark_point(size=20, color="red").encode(x="idx", y="w")
lines = alt.Chart(df_w).mark_line(size=1).encode(x="idx", y="w")

points + lines

alt.LayerChart(...)